# RegionAnalyzer — synthetic 3D segmentation

Build a small labeled volume (three non-overlapping objects) and extract region properties as a pandas DataFrame.

In [36]:
import numpy as np
import pandas as pd
import stackview

from vistiq.utils import ArrayIteratorConfig
from vistiq.segment.analysis import RegionAnalyzer, RegionAnalyzerConfig, region_to_numpy, dataframe_to_numpy
from vistiq.segment.select import RegionFilter, RegionFilterConfig, RangeFilterConfig, ValueFilter, ValueFilterConfig, TopKFilter, TopKFilterConfig, FULL, LOWER, UPPER, LOWER_ND, UPPER_ND, OFF_DIAGONAL

## Synthetic label volume

Shape `(100, 100, 10)` with labels `1`, `2`, and `3` in separate spatial regions (no overlap), mimicking a 3D segmentation mask.

In [37]:
labels = np.zeros((10, 200, 200), dtype=np.uint64)

# Object 1 — upper-left
labels[2:7, 20:38, 12:30] = 1

# Object 2 — center
labels[3:9, 42:58, 38:62] = 2

# Object 3 — lower-right
labels[1:6, 72:92, 68:88] = 3

# Object 4 — lower-right
labels[4:6, 112:132, 125:163] = 4

# Object 5 — lower-right
labels[7:9, 145:180, 12:58] = 5

unique_labels = np.unique(labels)
print(f"labels.shape={labels.shape}, dtype={labels.dtype}")
print(f"unique labels: {unique_labels}")
print(f"voxel counts: {{{', '.join(f'{int(l)}: {int((labels == l).sum())}' for l in unique_labels if l)}}}")

labels.shape=(10, 200, 200), dtype=uint64
unique labels: [0 1 2 3 4 5]
voxel counts: {1: 1620, 2: 2304, 3: 2000, 4: 1520, 5: 3220}


In [38]:
areas = np.zeros((10, 200, 200), dtype=np.uint64)

# Area 1 — upper-left
areas[2:9, 10:58, 10:98] = 6


# Area2 — lower-right
areas[1:10, 96:192, 58:178] = 8

unique_labels = np.unique(labels)
print(f"labels.shape={labels.shape}, dtype={labels.dtype}")
print(f"unique labels: {unique_labels}")
print(f"voxel counts: {{{', '.join(f'{int(l)}: {int((labels == l).sum())}' for l in unique_labels if l)}}}")

labels.shape=(10, 200, 200), dtype=uint64
unique labels: [0 1 2 3 4 5]
voxel counts: {1: 1620, 2: 2304, 3: 2000, 4: 1520, 5: 3220}


In [39]:
stackview.slice(np.concatenate([labels, areas], axis=-1))

## RegionAnalyzer (dataframe output)

Analyze the full 3D volume (`slice_def=()`). With `map_axes=True`, vector properties such as `cross_sectional_area` and `aspect_ratio` are expanded to plane-specific columns (`-xy`, `-xz`, `-yz`).

In [40]:
metadata = {
    "axes": ["Z", "Y", "X"],
    "scale": (2.0, 1.0, 1.0),
}

config = RegionAnalyzerConfig(
    output_type="dataframe",
    map_axes=True,
    properties=[
        "label",
        "volume",
        "centroid",
        "bbox",
        "aspect_ratio",
        "cross_sectional_area",
    ],
    iterator_config=ArrayIteratorConfig(slice_def=()),
)

l_regions = RegionAnalyzer(config).run(labels, metadata=metadata)
a_regions = RegionAnalyzer(config).run(areas, metadata=metadata)

#print(f"{len(l_regions)} regions, {len(l_regions.columns)} columns")
#print(f"{len(a_regions)} areas, {len(a_regions.columns)} columns")

2026-06-10 00:13:48,246 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 00:13:48,346 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 00:13:48,370 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='dataframe' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None properties=['label', 'stack_id', 'slice_id', 'object_id', 'centroid', 'bbox', 'aspect_ratio', 'cross_sectional_area', 'area'] map_axes=True expand_co

In [41]:
l_regions

,centroid-z,centroid-y,centroid-x,bbox-start-z,bbox-start-y,bbox-start-x,bbox-end-z,bbox-end-y,bbox-end-x,volume,aspect_ratio-yz,aspect_ratio-xz,aspect_ratio-xy,aspect_ratio,cross_sectional_area-yz,cross_sectional_area-xz,cross_sectional_area-xy,object_id,stack_id,slice_id
label,,,,,,,,,,,,,,,,,,,,
1,8.0,28.5,20.5,2,20,12,7,38,30,3240.0,0.545173,0.545173,1.000000,0.545173,180.0,180.0,324.0,6fe46a65675c43d19b81a0dc274c1013,4cdc0beb0ef44bcf87837ff54b37b8d8,5a0c38ae8e214ef5a5da5eb76d73e290
2,11.0,49.5,49.5,3,42,38,9,58,62,4608.0,0.740959,0.493435,0.665942,0.493435,192.0,288.0,384.0,411efa7b369842f5a43200d974210672,4cdc0beb0ef44bcf87837ff54b37b8d8,5a0c38ae8e214ef5a5da5eb76d73e290
3,6.0,81.5,77.5,1,72,68,6,92,88,4000.0,0.490511,0.490511,1.000000,0.490511,200.0,200.0,400.0,d1044f4c1d1f45c2b680834ae5c15f61,4cdc0beb0ef44bcf87837ff54b37b8d8,5a0c38ae8e214ef5a5da5eb76d73e290
4,9.0,121.5,143.5,4,112,125,6,132,163,3040.0,0.173422,0.091192,0.525840,0.091192,80.0,152.0,760.0,716ba7f6a10a4f31b61d7874940f9ad0,4cdc0beb0ef44bcf87837ff54b37b8d8,5a0c38ae8e214ef5a5da5eb76d73e290
5,15.0,162.0,34.5,7,145,12,9,180,58,6440.0,0.099015,0.075324,0.760739,0.075324,140.0,184.0,1610.0,03a59f1a8a90462294d81e96a6998cd5,4cdc0beb0ef44bcf87837ff54b37b8d8,5a0c38ae8e214ef5a5da5eb76d73e290


In [42]:
a_regions

,centroid-z,centroid-y,centroid-x,bbox-start-z,bbox-start-y,bbox-start-x,bbox-end-z,bbox-end-y,bbox-end-x,volume,aspect_ratio-yz,aspect_ratio-xz,aspect_ratio-xy,aspect_ratio,cross_sectional_area-yz,cross_sectional_area-xz,cross_sectional_area-xy,object_id,stack_id,slice_id
label,,,,,,,,,,,,,,,,,,,,
6,10.0,33.5,53.5,2,10,10,9,58,98,59136.0,0.288738,0.157469,0.545371,0.157469,672.0,1232.0,4224.0,cdabe89011e94019920d0391cee72729,893ae0c1dccb429fa1fa6fb900bb4b53,aa99f03624074a7ea2969d69ad2ae0f3
8,10.0,143.5,117.5,1,96,58,10,192,178,207360.0,0.186349,0.149076,0.799984,0.149076,1728.0,2160.0,11520.0,ffc454ae2b8c42af8653a030c29ceab6,893ae0c1dccb429fa1fa6fb900bb4b53,aa99f03624074a7ea2969d69ad2ae0f3


In [43]:
rfcfg = RegionFilterConfig(
    filters=[
        RangeFilterConfig(
            attribute="volume",
            range=(100.0,np.inf)
        ),
        #MinFilterConfig(
        #    attribute="aspect_ratio",
        #    minimum=0.015,
        #),
    ]
)
l_accepted, _ = RegionFilter(rfcfg).run(l_regions)
a_accepted, _ = RegionFilter(rfcfg).run(a_regions)

2026-06-10 00:13:50,147 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 00:13:50,202 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 00:13:50,209 - INFO - Running RegionFilter with config: classname='Configurable' package='vistiq.core' version=None command_group=None filters=[RangeFilterConfig(classname='Configurable', package='vistiq.core', version=None, command_group=None, attribute='volume', axis=None, strict=True, preferred_input_type='numpy', range=(100.0, inf))]
2026-06-10 00:13:50,210 - INFO - Applying RegionFilter to a DataFrame
2026-06-10 00:13:50,211 - INFO - RegionFilter: len(accepted_regions)=5, len(removed_labels)=0
2026-06-10 00:13:50,213 - INFO - Finished in state Comple

In [44]:
l_accepted

,centroid-z,centroid-y,centroid-x,bbox-start-z,bbox-start-y,bbox-start-x,bbox-end-z,bbox-end-y,bbox-end-x,volume,aspect_ratio-yz,aspect_ratio-xz,aspect_ratio-xy,aspect_ratio,cross_sectional_area-yz,cross_sectional_area-xz,cross_sectional_area-xy,object_id,stack_id,slice_id
label,,,,,,,,,,,,,,,,,,,,
1,8.0,28.5,20.5,2,20,12,7,38,30,3240.0,0.545173,0.545173,1.000000,0.545173,180.0,180.0,324.0,6fe46a65675c43d19b81a0dc274c1013,4cdc0beb0ef44bcf87837ff54b37b8d8,5a0c38ae8e214ef5a5da5eb76d73e290
2,11.0,49.5,49.5,3,42,38,9,58,62,4608.0,0.740959,0.493435,0.665942,0.493435,192.0,288.0,384.0,411efa7b369842f5a43200d974210672,4cdc0beb0ef44bcf87837ff54b37b8d8,5a0c38ae8e214ef5a5da5eb76d73e290
3,6.0,81.5,77.5,1,72,68,6,92,88,4000.0,0.490511,0.490511,1.000000,0.490511,200.0,200.0,400.0,d1044f4c1d1f45c2b680834ae5c15f61,4cdc0beb0ef44bcf87837ff54b37b8d8,5a0c38ae8e214ef5a5da5eb76d73e290
4,9.0,121.5,143.5,4,112,125,6,132,163,3040.0,0.173422,0.091192,0.525840,0.091192,80.0,152.0,760.0,716ba7f6a10a4f31b61d7874940f9ad0,4cdc0beb0ef44bcf87837ff54b37b8d8,5a0c38ae8e214ef5a5da5eb76d73e290
5,15.0,162.0,34.5,7,145,12,9,180,58,6440.0,0.099015,0.075324,0.760739,0.075324,140.0,184.0,1610.0,03a59f1a8a90462294d81e96a6998cd5,4cdc0beb0ef44bcf87837ff54b37b8d8,5a0c38ae8e214ef5a5da5eb76d73e290


In [45]:
a_accepted

,centroid-z,centroid-y,centroid-x,bbox-start-z,bbox-start-y,bbox-start-x,bbox-end-z,bbox-end-y,bbox-end-x,volume,aspect_ratio-yz,aspect_ratio-xz,aspect_ratio-xy,aspect_ratio,cross_sectional_area-yz,cross_sectional_area-xz,cross_sectional_area-xy,object_id,stack_id,slice_id
label,,,,,,,,,,,,,,,,,,,,
6,10.0,33.5,53.5,2,10,10,9,58,98,59136.0,0.288738,0.157469,0.545371,0.157469,672.0,1232.0,4224.0,cdabe89011e94019920d0391cee72729,893ae0c1dccb429fa1fa6fb900bb4b53,aa99f03624074a7ea2969d69ad2ae0f3
8,10.0,143.5,117.5,1,96,58,10,192,178,207360.0,0.186349,0.149076,0.799984,0.149076,1728.0,2160.0,11520.0,ffc454ae2b8c42af8653a030c29ceab6,893ae0c1dccb429fa1fa6fb900bb4b53,aa99f03624074a7ea2969d69ad2ae0f3


In [46]:
from vistiq.analysis import DistanceCalculator, DistanceCalculatorConfig

dccfg = DistanceCalculatorConfig(
    annotate=True, 
    output_type="torch.Tensor"
)

centroids = dataframe_to_numpy(l_accepted, attributes=["centroid"], strict=False)
object_ids = dataframe_to_numpy(l_accepted, attributes=["object_id"])
dist = DistanceCalculator(dccfg).run(centroids, centroids, spacing=metadata.get("scale", None), point_annotations=(object_ids, object_ids))

2026-06-10 00:13:52,921 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 00:13:52,970 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 00:13:52,977 - INFO - Found mps device: Apple Metal (MPS)
2026-06-10 00:13:52,985 - INFO - MatrixCalculator.run: device=mps, points1.shape=torch.Size([5, 3]), points1.dtype=torch.float32, points2.shape=torch.Size([5, 3]), points2.dtype=torch.float32, spacing=(2.0, 1.0, 1.0)
2026-06-10 00:13:52,986 - INFO - DistanceCalculator._calculate: distances.shape=torch.Size([5, 5])
2026-06-10 00:13:52,988 - INFO - Finished in state Completed()


In [47]:
type(dist), dist

(torch.Tensor,
 tensor([[  0.0000,  36.3043,  77.9359, 154.2141, 134.9602],
         [ 36.3043,   0.0000,  43.6807, 118.4736, 113.7772],
         [ 77.9359,  43.6807,   0.0000,  77.4080,  93.0229],
         [154.2141, 118.4736,  77.4080,   0.0000, 116.8985],
         [134.9602, 113.7772,  93.0229, 116.8985,   0.0000]], device='mps:0'))

In [48]:
from vistiq.segment import UPPER, DIAGONAL
tkcfg = TopKFilterConfig(
    k=1,
    axis=1,
    largest=False,
    triangle=OFF_DIAGONAL,
    output="masked_values",
)

tk = TopKFilter(tkcfg).run(dist)
tk

2026-06-10 00:13:55,286 - INFO - Found mps device: Apple Metal (MPS)


tensor([[    nan, 36.3043,     nan,     nan,     nan],
        [36.3043,     nan,     nan,     nan,     nan],
        [    nan, 43.6807,     nan,     nan,     nan],
        [    nan,     nan, 77.4080,     nan,     nan],
        [    nan,     nan, 93.0229,     nan,     nan]], device='mps:0')

In [49]:
import torch

mincfg = ValueFilterConfig(
    ref_value=80.0,
    axis=0,
    operator=">",
    triangle=LOWER_ND,
    output="masked_values",
)
maxcfg = ValueFilterConfig(
    ref_value=120.0,
    axis=0,
    operator="<",
    triangle=LOWER_ND,
    output="masked_values",
)
mint = ValueFilter(mincfg).run(dist)
maxt = ValueFilter(maxcfg).run(dist)
ranget = torch.eq(mint,maxt)
mint, maxt, ranget

2026-06-10 00:13:56,237 - INFO - Found mps device: Apple Metal (MPS)
2026-06-10 00:13:56,243 - INFO - Found mps device: Apple Metal (MPS)


(tensor([[     nan,      nan,      nan,      nan,      nan],
         [     nan,      nan,      nan,      nan,      nan],
         [     nan,      nan,      nan,      nan,      nan],
         [154.2141, 118.4736,      nan,      nan,      nan],
         [134.9602, 113.7772,  93.0229, 116.8985,      nan]], device='mps:0'),
 tensor([[     nan,      nan,      nan,      nan,      nan],
         [ 36.3043,      nan,      nan,      nan,      nan],
         [ 77.9359,  43.6807,      nan,      nan,      nan],
         [     nan, 118.4736,  77.4080,      nan,      nan],
         [     nan, 113.7772,  93.0229, 116.8985,      nan]], device='mps:0'),
 tensor([[False, False, False, False, False],
         [False, False, False, False, False],
         [False, False, False, False, False],
         [False,  True, False, False, False],
         [False,  True,  True,  True, False]], device='mps:0'))

In [50]:
from vistiq.analysis.matrix import MatrixAggregatorConfig, MatrixAggregator

macfg = MatrixAggregatorConfig(
    operation="sum",
    axis=1,
)

counts = MatrixAggregator(macfg).run(ranget>0)
counts

2026-06-10 00:14:01,737 - INFO - Found mps device: Apple Metal (MPS)


tensor([0, 0, 0, 1, 3], device='mps:0')

# Build graph

In [ ]:
import networkx as nx
from pyvis.network import Network

G = nx.from_pandas_adjacency(dist)

In [ ]:
net = Network(notebook=True, cdn_resources='remote', bgcolor="#222222", font_color="white", select_menu=True)
net.barnes_hut()

# Convert the networkx object
net.from_nx(G)
neighbor_map = net.get_adj_list()

# add neighbor data to node hover data
for node in net.nodes:
    node["title"] = node["id"] + "\n" +"  Neighbors:\n" + "\n".join(neighbor_map[node["id"]])
    node["value"] = len(neighbor_map[node["id"]])

# Render
net.show("nx_graph.html")

In [ ]:
net.show_buttons(filter_=['physics'])